# Pipeline do modelo - casos de teste

Fluxo: `cases.csv` -> split treino/teste (por artigo) -> pre-processamento -> rotulacao fraca (treino, MeSH) -> features por token + regressao logistica -> predicao no teste -> pos-processamento.

Estrutura de pastas esperada:

```
project02/
  data/
    raw/sample/cases.csv
    external/MeSH/       mesh_anatomy.csv, mesh_diseases.csv, mesh_*.csv
  models/
    pre_processamento/   tokenizer.py, split_cases.py
    classificar_nos/     step2_weak_labels.py, features.py, ...
  notebook/              pipeline_modelo.ipynb   <- este arquivo
```

Rode o notebook com o diretorio de trabalho em `project02/notebook`.

## 1. Caminhos e imports

In [11]:
# Recarrega automaticamente os .py de project02/ quando voce os edita (evita ficar com versao antiga em memoria)
%load_ext autoreload
%autoreload 2

import sys
import json
from pathlib import Path

import pandas as pd

# project02/notebook -> project02 -> pasta que CONTEM project02 (necessaria para `import project02...`)
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_DIR = NOTEBOOK_DIR.parent
ROOT = PROJECT_DIR.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CASES_CSV = PROJECT_DIR / 'data' / 'raw' / 'sample' / 'cases.csv'
PRE_DIR = PROJECT_DIR / 'models' / 'pre_processamento'
NOS_DIR = PROJECT_DIR / 'models' / 'classificar_nos'
SPLITS_DIR = PROJECT_DIR / 'data' / 'processed' / 'splits'
OUT_DIR = PROJECT_DIR / 'data' / 'processed'

# confere se tudo esta no lugar
esperados = [
    CASES_CSV,
    PRE_DIR / 'tokenizer.py',
    PRE_DIR / 'split_cases.py',
    NOS_DIR / 'step2_weak_labels.py',
    NOS_DIR / 'features.py',
]
for p in esperados:
    print('OK     ' if p.exists() else 'FALTA  ', p.relative_to(ROOT))

# confere se o tokenizer.py e a versao nova (com os tokens com posicao, usados na rotulacao fraca)
if 'PREPROC_ID' not in (PRE_DIR / 'tokenizer.py').read_text(encoding='utf-8'):
    print('\nATENCAO: tokenizer.py esta desatualizado (sem PREPROC_ID). Substitua pelo arquivo novo.')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
OK      project02/data/raw/sample/cases.csv
OK      project02/models/pre_processamento/tokenizer.py
OK      project02/models/pre_processamento/split_cases.py
OK      project02/models/classificar_nos/step2_weak_labels.py
OK      project02/models/classificar_nos/features.py


## 2. Parametros

Tudo definido aqui, sem arquivo de configuracao externo.

In [12]:
TEST_SIZE = 0.2          # fracao dos artigos que vai para o teste
SEED = 42                # para reproduzir o mesmo split
GROUP_COL = 'article_id' # casos do mesmo artigo ficam juntos (evita vazamento)
TEXT_COL = 'case_text'   # coluna com o texto do caso; None = detecta a coluna de texto mais longa

# Rotulacao fraca (passo 2)
MESH_DIR = PROJECT_DIR / 'data' / 'external' / 'MeSH'                          # mesh_*.csv
STOPLIST = PROJECT_DIR / 'models' / 'classificar_nos' / 'stoplist.txt'         # opcional: termos a ignorar, um por linha
WEAK_DIR = PROJECT_DIR / 'data' / 'processed' / 'weak_labels'                  # saida

# Modelo (passo 3)
DICT_DROPOUT = 0.5       # chance de esconder cada termo do dicionario no treino (0 = nunca, 1 = sempre)
WINDOW = 2               # vizinhos de cada lado usados como contexto
N_FOLDS = 5              # validacao cruzada por artigo, dentro do treino
MODEL_DIR = PROJECT_DIR / 'data' / 'processed' / 'model'                       # saida

## 3. Split treino / teste (agrupado por artigo)

In [13]:
from project02.models.pre_processamento.split_cases import split_cases

train, test = split_cases(
    cases_csv=CASES_CSV,
    out_dir=SPLITS_DIR,
    test_size=TEST_SIZE,
    seed=SEED,
    group_col=GROUP_COL,
)

print(f'treino: {len(train)} casos / {train[GROUP_COL].nunique()} artigos')
print(f'teste : {len(test)} casos / {test[GROUP_COL].nunique()} artigos')
print('salvo em', SPLITS_DIR.relative_to(ROOT))
print('colunas:', list(test.columns))
test.head(3)

treino: 45 casos / 40 artigos
teste : 11 casos / 10 artigos
salvo em project02/data/processed/splits
colunas: ['article_id', 'age', 'case_id', 'case_text', 'gender']


,article_id,age,case_id,case_text,gender
0,PMC5137649,44.0,PMC5137649_01,A 44-year-old woman presented with a 3-day his...,Female
8,PMC5833205,0.0,PMC5833205_01,A previously healthy 20-month-old boy presente...,Male
13,PMC9529523,41.0,PMC9529523_01,A 41-year-old African American female with an ...,Female


## 4. Pre-processamento do teste

Duas versoes de tokens: `tokens_lower` (minusculas, sem pontuacao) e `tokens_stem` (com radical).

In [14]:
from project02.models.pre_processamento.tokenizer import preprocess

if TEXT_COL is None:
    # is_string_dtype cobre tanto 'object' quanto o tipo 'str' das versoes novas do pandas
    ignorar = {'case_id', GROUP_COL}
    candidatas = [c for c in test.columns if pd.api.types.is_string_dtype(test[c]) and c not in ignorar]
    TEXT_COL = max(candidatas, key=lambda c: test[c].astype(str).str.len().mean())
print('coluna de texto:', TEXT_COL)

test = test.copy()
textos = test[TEXT_COL].fillna('').astype(str)
test['tokens_lower'] = textos.apply(lambda t: preprocess(t, use_stem=False))
test['tokens_stem'] = textos.apply(lambda t: preprocess(t, use_stem=True))

test[[TEXT_COL, 'tokens_lower', 'tokens_stem']].head(2)

coluna de texto: case_text


,case_text,tokens_lower,tokens_stem
0,A 44-year-old woman presented with a 3-day his...,"[a, 44-year-old, woman, presented, with, a, 3-...","[a, 44-year-old, woman, present, with, a, 3-da..."
8,A previously healthy 20-month-old boy presente...,"[a, previously, healthy, 20-month-old, boy, pr...","[a, previous, healthi, 20-month-old, boy, pres..."


## 5. Rotulacao fraca (treino)

Usa os dicionarios MeSH (`data/external/MeSH/mesh_*.csv`) para marcar, em cada caso de **treino**, os trechos
de Disease, Anatomy, Symptom, Medication etc. no formato BIO (match do termo mais longo). `Patient`
(idade e genero) sai de regras. O teste nao e rotulado aqui: ele e anotado a mao para medir o modelo.

Se algum `mesh_*.csv` nao existir, o tipo correspondente fica sem rotulos (o aviso aparece abaixo).

In [15]:
from project02.models.classificar_nos.step2_weak_labels import (
    build_dictionary, label_dataframe, save_weak_labels, summarize,
)

# 1) dicionario MeSH normalizado do mesmo jeito que o texto (minusculas, sem pontuacao, radical)
dic = build_dictionary(MESH_DIR, stoplist=STOPLIST)

# 2) tokeniza os casos de TREINO (com posicao no texto) e rotula pelo dicionario
records, metas, n_tokens = label_dataframe(
    train, dic, text_col=TEXT_COL, id_col='case_id', group_col=GROUP_COL,
)

# 3) salva weak_labels.jsonl e dictionary.json
weak_path = save_weak_labels(records, dic, WEAK_DIR)
print('salvo em', weak_path.relative_to(ROOT))

summarize(records, metas, n_tokens)

Dicionarios carregados: Anatomy, Disease, Exam, Medication, Result, Symptom, Procedure
Dicionario: 14165 termos normalizados (824 ambiguos, 1265 IGNORE)
salvo em project02/data/processed/weak_labels/weak_labels.jsonl

Casos: 45 | tokens: 26801 | tokens em span/IGNORE: 2349 (8.8%)
Spans por rotulo principal: {'Anatomy': 530, 'Patient': 331, 'Disease': 297, 'Exam': 234, 'IGNORE': 171, 'Medication': 145, 'Symptom': 141, 'Result': 69}
Spans ambiguos (candidates): {'Disease|Result': 46, 'Symptom|Disease': 31, 'Symptom|Result': 4, 'Symptom|Exam': 2, 'Symptom|Disease|Result': 1}
Patient/genero vs CSV: ok=41 divergente=1 nao_extraido=0
Patient/idade  vs CSV: ok=37 divergente=1 nao_extraido=2


In [16]:
# Revisao: quais termos mais geram span em cada rotulo? (ajuda a achar ruido para a stoplist)
todos = pd.DataFrame([dict(s, case_id=r['case_id']) for r in records for s in r['spans']])

if len(todos):
    topo = (todos.assign(termo=todos['text'].str.lower())
                 .groupby('label')['termo'].value_counts()
                 .groupby(level=0).head(8))
    display(topo.to_frame('n'))
else:
    print('Nenhum span encontrado: confira MESH_DIR e o texto dos casos.')

# Exemplo de um caso
r0 = records[0]
print(r0['case_id'], r0['patient'])
pd.DataFrame(r0['spans'])[['text', 'label', 'candidates', 'source']]

n
label   termo          
Anatomy blood        48
        abdomen      20
        tissue       13
        colon        12
        lung         11
...                  ..
Symptom chest pain   10
        edema         6
        diarrhea      5
        nausea        5
        weight loss   5

[64 rows x 1 columns]

PMC9387390_01 {'age_years': 57.0, 'gender': 'male', 'gender_source': 'head', 'has_mention': True}


,text,label,candidates,source
0,man,Patient,[Patient],rule
1,diarrhea,Symptom,[Symptom],dict
2,tomography,Exam,[Exam],dict
3,edema,Symptom,[Symptom],dict
4,barium enema,Exam,[Exam],dict
5,abdomen,Anatomy,[Anatomy],dict
6,pancreas,Anatomy,[Anatomy],dict
7,duodenum,Anatomy,[Anatomy],dict
8,patient,Patient,[Patient],rule
9,biopsy,Exam,[Exam],dict


## 6. Modelo: features por token + regressao logistica

Cada token vira um dicionario de features (forma da palavra, vizinhos numa janela, palavras de contexto e o
tipo dos termos do MeSH que aparecem ao redor) -> `DictVectorizer` -> `LogisticRegression` multinomial. Sem embeddings.

Dois cuidados para o modelo nao virar apenas uma copia do dicionario (ver `features.py`):

- **Dropout do dicionario:** no treino, cada termo do MeSH e escondido com probabilidade `DICT_DROPOUT`, entao o
  modelo tambem aprende pela forma da palavra e pelo contexto.
- **Termos ambiguos** (ex.: Disease|Result) ficam fora do treino, porque o rotulo fraco deles vem de uma ordem de
  prioridade, nao do contexto. Na predicao o contexto escolhe o tipo entre os candidatos do dicionario.

**Atencao:** a validacao abaixo compara com os rotulos *fracos*, que nascem do proprio dicionario. Serve para comparar
configuracoes (janela, dropout, C), nao como medida final. A medida final e no teste anotado a mao.

In [17]:
import numpy as np
from sklearn.metrics import classification_report
from sklearn.model_selection import GroupKFold

from project02.models.classificar_nos.features import build_dataset, make_model

# Tres visoes do MESMO conjunto de tokens (mesma ordem): dicionario escondido em parte dos termos (treino),
# dicionario visivel (como na predicao) e dicionario todo escondido (mede se o modelo generaliza sem ele).
kw = dict(text_col=TEXT_COL, id_col='case_id', window=WINDOW)
X_tr, y, ids = build_dataset(train, records, dict_dropout=DICT_DROPOUT, seed=SEED, **kw)
X_com, _, _ = build_dataset(train, records, dict_dropout=0.0, **kw)
X_sem, _, _ = build_dataset(train, records, dict_dropout=1.0, **kw)
y = np.array(y)

artigo = dict(zip(train['case_id'].astype(str), train[GROUP_COL]))
grupos = np.array([artigo[cid] for cid, _ in ids])
print(f'tokens de treino: {len(y)} | artigos: {len(set(grupos))}')

# Validacao cruzada por artigo: casos do mesmo artigo nunca ficam em treino e validacao ao mesmo tempo
n_splits = min(N_FOLDS, len(set(grupos)))
pred_com = np.empty(len(y), dtype=object)
pred_sem = np.empty(len(y), dtype=object)
for tr_idx, va_idx in GroupKFold(n_splits=n_splits).split(np.zeros(len(y)), y, grupos):
    m = make_model().fit([X_tr[k] for k in tr_idx], y[tr_idx])
    pred_com[va_idx] = m.predict([X_com[k] for k in va_idx])
    pred_sem[va_idx] = m.predict([X_sem[k] for k in va_idx])

print('=== COM dicionario (como na predicao) ===')
print(classification_report(y, pred_com, zero_division=0))
print('=== SEM dicionario (generalizacao: so forma da palavra e contexto) ===')
print(classification_report(y, pred_sem, zero_division=0))

tokens de treino: 26483 | artigos: 40
=== COM dicionario (como na predicao) ===
              precision    recall  f1-score   support

   B-Anatomy       0.87      1.00      0.93       530
   B-Disease       0.79      0.99      0.88       251
      B-Exam       0.87      1.00      0.93       234
B-Medication       0.79      0.96      0.86       145
   B-Patient       0.98      0.97      0.98       331
    B-Result       0.94      0.87      0.90        69
   B-Symptom       0.96      0.97      0.97       103
   I-Anatomy       0.83      0.97      0.89       100
   I-Disease       0.82      0.94      0.88        86
      I-Exam       0.81      0.98      0.89       126
I-Medication       0.71      0.38      0.50        13
   I-Patient       1.00      0.56      0.71         9
    I-Result       1.00      0.25      0.40         4
   I-Symptom       1.00      0.93      0.97        30
           O       1.00      0.99      0.99     24452

    accuracy                           0.99     26483


In [18]:
import joblib

# Modelo final: treina com todo o treino
model = make_model().fit(X_tr, y)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODEL_DIR / 'model.joblib')
print('modelo salvo em', (MODEL_DIR / 'model.joblib').relative_to(ROOT))

# O que o modelo aprendeu: as features de maior peso em cada classe
vec, clf = model.steps[0][1], model.steps[1][1]
nomes = np.array(vec.get_feature_names_out())
for k, classe in enumerate(clf.classes_):
    if classe == 'O':
        continue
    top = nomes[np.argsort(clf.coef_[k])[-8:][::-1]]
    print(f'{classe:13}', ', '.join(top))

modelo salvo em project02/data/processed/model/model.joblib
B-Anatomy     dict:Anatomy, suf2=um, dict_first, -2:dict:Anatomy, -2:norm=the, -1:suf3=ght, pre3=lym, norm=neutrophil
B-Disease     dict:Disease, norm=infect, dict_first, lower=infection, suf2=al, pre3=pne, suf2=ia, suf2=is
B-Exam        dict:Exam, suf4=aphy, dict_first, pre3=bio, norm=dilat, pre3=dil, suf3=phy, +1:dict:Exam
B-Medication  dict:Medication, suf2=in, suf2=ne, dict_first, ctx:mg, suf3=cin, suf3=rin, suf3=ine
B-Patient     norm=patient, pre3=pat, suf4=male, suf3=ale, lower=patient, prev+cur=the|patient, suf2=le, suf4=ient
B-Result      dict:Result, norm=fistula, pre3=fis, pre3=hem, norm=ulcer, pre3=ulc, dict_first, pre3=inf
B-Symptom     dict:Symptom, dict_first, ctx:pain, pre3=fev, norm=fever, +1:suf3=ain, +1:norm=pain, suf2=ea
I-Anatomy     -1:dict:Anatomy, dict:Anatomy, dict_inside, norm=extrem, pre3=ext, suf2=ve, -1:suf3=ary, -1:suf3=nal
I-Disease     -1:dict:Disease, dict:Disease, dict_inside, ctx:diagnos, -1:

## 7. Predicao no teste e pos-processamento

Para cada caso de teste: casa o texto com o dicionario (so para obter os candidatos), prediz as etiquetas BIO,
corrige BIO invalido (um `I-X` sem `B-X` antes vira `B-X`) e converte em entidades. Nos termos ambiguos o
contexto escolhe o tipo entre os candidatos, que tambem ficam guardados em `candidates`.

In [19]:
from project02.models.pre_processamento.tokenizer import process_case
from project02.models.classificar_nos.step2_weak_labels import label_case
from project02.models.classificar_nos.features import predict_case, fix_bio, bio_to_entities

entidades = []
for row in test.to_dict('records'):
    case = process_case(str(row['case_id']), str(row[GROUP_COL]), str(row[TEXT_COL]))
    record = label_case(case, dic)                        # aqui so interessa o casamento com o dicionario
    tags, cands = predict_case(model, case, record, window=WINDOW)
    tags = fix_bio(tags)                                  # pos-processamento: I- sem B- vira B-
    entidades.append(bio_to_entities(case, tags, cands))

test['entities'] = entidades
print('casos processados:', len(test))
print('casos com alguma entidade:', int((test['entities'].str.len() > 0).sum()))
print('entidades por tipo:', pd.Series([e['label'] for ents in entidades for e in ents]).value_counts().to_dict())

# Exemplo: primeiro caso do teste
ex = entidades[0]
pd.DataFrame(ex)[['text', 'label', 'candidates']] if ex else 'nenhuma entidade no primeiro caso'

casos processados: 11
casos com alguma entidade: 11
entidades por tipo: {'Anatomy': 111, 'Patient': 109, 'Disease': 95, 'Exam': 75, 'Medication': 30, 'Symptom': 28, 'Result': 10}


,text,label,candidates
0,woman,Patient,[]
1,abdominal pain,Symptom,[Symptom]
2,nausea,Symptom,[Symptom]
3,constipation,Symptom,[Symptom]
4,physical examination,Exam,[Exam]
5,tomography,Exam,[Exam]
6,stomach,Anatomy,[Anatomy]
7,pancreas,Anatomy,[Anatomy]
8,pancreatic,Disease,[Disease]
9,cyst,Disease,"[Disease, Result]"


## 8. Salvar predicoes

In [20]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

saida = test.copy()
for col in ['tokens_lower', 'tokens_stem', 'entities']:
    saida[col] = saida[col].apply(json.dumps)

destino = OUT_DIR / 'test_predictions.csv'
saida.to_csv(destino, index=False)
print('salvo em', destino.relative_to(ROOT))

salvo em project02/data/processed/test_predictions.csv


## 9. Rotulacao fraca de arestas (treino) - Passo 1

Para cada sentenca dos casos de **treino**, gera os pares candidatos `(E1, E2)` + o texto intermediario (`span_between`) e atribui o rotulo fraco da aresta (`FAMILIAR_HISTORY`, `HISTORY`, `DIAGNOSED_WITH`, `PRESENTS_WITH`, `TAKES`, `UNDERWENT_EXAM`, `LOCATED_IN`, `REVEALS`, `HAS_EFFECT` ou `NONE`).

In [21]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from models.classificar_arestas.step1_weak_edges import (
    label_edges_dataframe,
    save_weak_edges,
    summarize_edges,
)

WEAK_EDGES_DIR = PROJECT_DIR / 'data' / 'processed' / 'weak_edges'

# 1) Gera pares candidatos (E1, E2) por sentenca no treino e rotula fracamente
edge_records_train = label_edges_dataframe(
    train,
    records,
    text_col=TEXT_COL,
    id_col='case_id',
    group_col=GROUP_COL,
)

# 2) Salva weak_edges.jsonl e exibe resumo
weak_edges_path = save_weak_edges(edge_records_train, WEAK_EDGES_DIR)
print('salvo em', weak_edges_path.relative_to(ROOT))
print()
summarize_edges(edge_records_train)

# 3) Visualiza exemplos de pares candidatos gerados no treino
df_edges_preview = pd.DataFrame(edge_records_train)[
    [
        'case_id',
        'e1_text',
        'type_E1',
        'span_between',
        'e2_text',
        'type_E2',
        'token_dist',
        'label',
    ]
]
display(df_edges_preview[df_edges_preview['label'] != 'NONE'].head(15))
display(df_edges_preview[df_edges_preview['label'] == 'NONE'].head(10))

salvo em data/processed/weak_edges/weak_edges.jsonl

Casos com pares: 45 | Total de pares candidatos: 2039
Arestas positivas: 786 (38.5%) | NONE (sem relacao): 1253 (61.5%)
Pares por rotulo de aresta: {'NONE': 1253, 'DIAGNOSED_WITH': 210, 'UNDERWENT_EXAM': 208, 'TAKES': 139, 'PRESENTS_WITH': 104, 'LOCATED_IN': 79, 'HISTORY': 33, 'REVEALS': 7, 'FAMILIAR_HISTORY': 6}
Arestas positivas por par de tipos: {'Patient->Disease': 249, 'Patient->Exam': 208, 'Patient->Medication': 139, 'Patient->Symptom': 104, 'Disease->Anatomy': 55, 'Symptom->Anatomy': 24, 'Exam->Result': 7}


,case_id,e1_text,type_E1,span_between,e2_text,type_E2,token_dist,label
1,PMC9387390_01,Patient,Patient,Computed,tomography,Exam,1,UNDERWENT_EXAM
2,PMC9387390_01,Patient,Patient,Computed tomography CT imaging showed nonspeci...,edema,Symptom,7,PRESENTS_WITH
4,PMC9387390_01,Patient,Patient,An upper gastrointestinal UGI contrast study and,barium enema,Exam,7,UNDERWENT_EXAM
8,PMC9387390_01,patient,Patient,underwent CT guided,biopsy,Exam,3,UNDERWENT_EXAM
9,PMC9387390_01,patient,Patient,underwent CT guided biopsy of the mass this wa...,adenocarcinoma,Disease,16,DIAGNOSED_WITH
12,PMC9387390_01,adenocarcinoma,Disease,thought to have arisen from the,upper gastrointestinal tract,Anatomy,6,LOCATED_IN
13,PMC9387390_01,Patient,Patient,Approximately 11 months after the initiation o...,nausea,Symptom,14,PRESENTS_WITH
14,PMC9387390_01,Patient,Patient,His imaging was consistent with gastroduodenal...,pancreatic,Disease,11,DIAGNOSED_WITH
15,PMC9387390_01,Patient,Patient,Sutures placed at the GJ were noted to be loos...,abscesses,Disease,14,DIAGNOSED_WITH
18,PMC9387390_01,Patient,Patient,The efferent limb was subsequently,intubated,Exam,5,UNDERWENT_EXAM


,case_id,e1_text,type_E1,span_between,e2_text,type_E2,token_dist,label
0,PMC9387390_01,man,Patient,with no significant past medical or surgical h...,diarrhea,Symptom,15,NONE
3,PMC9387390_01,tomography,Exam,CT imaging showed nonspecific mesenteric,edema,Symptom,5,NONE
5,PMC9387390_01,abdomen,Anatomy,demonstrated a central mesenteric mass abuttin...,pancreas,Anatomy,7,NONE
6,PMC9387390_01,abdomen,Anatomy,demonstrated a central mesenteric mass abuttin...,duodenum,Anatomy,22,NONE
7,PMC9387390_01,pancreas,Anatomy,encasing the superior mesenteric vessels causi...,duodenum,Anatomy,14,NONE
10,PMC9387390_01,biopsy,Exam,of the mass this was consistent with invasive ...,adenocarcinoma,Disease,12,NONE
11,PMC9387390_01,biopsy,Exam,of the mass this was consistent with invasive ...,upper gastrointestinal tract,Anatomy,19,NONE
16,PMC9387390_01,duodenum,Anatomy,and proximal,jejunum,Anatomy,2,NONE
17,PMC9387390_01,stomach,Anatomy,and proximal,small intestine,Anatomy,2,NONE
21,PMC3437073_01,immobile,Exam,and attached to the,skin,Anatomy,4,NONE


## 10. Modelo de Arestas: PMI + TF-IDF (texto intermediario) + Distancia + Regressao Logistica Multinomial

Para cada par candidato `(E1, E2)` na mesma sentenca, construimos um vetor numerico combinando:
- **Tipos das entidades:** `type_E1`, `type_E2`, `pair=type_E1->type_E2` e `reading_order`
- **N-gramas / TF-IDF do texto entre elas (`span_between`)** e da janela de contexto local
- **Metrica PMI (Pointwise Mutual Information):** $\text{PMI}(E_1, E_2) = \log_2 \frac{P(E_1, E_2)}{P(E_1)P(E_2)}$
- **Distancia:** `token_dist` (numero de palavras entre $E_1$ e $E_2$), `log_dist` e barreira de oracao.

No treino, aplicamos `CUE_DROPOUT = 0.5` (esconde palavras-gatilho em 50% das vezes) para que o modelo aprenda tambem pelo contexto, distancia e PMI.

In [22]:
from project02.models.classificar_arestas.edge_features import (
    build_edge_dataset,
    make_edge_model,
)
from sklearn.metrics import classification_report
from sklearn.model_selection import GroupKFold

CUE_DROPOUT = 0.5

# Tres visoes do MESMO conjunto de pares candidatos de treino:
# 1) X_edge_tr: com dropout de pistas (treino)
# 2) X_edge_com: com todas as palavras visiveis (como na predicao)
# 3) X_edge_sem: sem as palavras-gatilho explicitas (mede generalizacao por TF-IDF residual, PMI, tipos e distancia)
X_edge_tr, y_edge, edge_ids = build_edge_dataset(
    edge_records_train, cue_dropout=CUE_DROPOUT, seed=SEED
)
X_edge_com, _, _ = build_edge_dataset(
    edge_records_train, cue_dropout=0.0, seed=SEED
)
X_edge_sem, _, _ = build_edge_dataset(
    edge_records_train, cue_dropout=1.0, seed=SEED
)

grupos_edge = np.array([artigo[cid] for cid, _ in edge_ids])
print(
    f'pares de treino: {len(y_edge)} | artigos: {len(set(grupos_edge))}'
)

n_splits_edge = min(N_FOLDS, len(set(grupos_edge)))
pred_edge_com = np.empty(len(y_edge), dtype=object)
pred_edge_sem = np.empty(len(y_edge), dtype=object)

for tr_idx, va_idx in GroupKFold(n_splits=n_splits_edge).split(
    np.zeros(len(y_edge)), y_edge, grupos_edge
):
  m_edge = make_edge_model(seed=SEED).fit(
      [X_edge_tr[k] for k in tr_idx], y_edge[tr_idx]
  )
  pred_edge_com[va_idx] = m_edge.predict([X_edge_com[k] for k in va_idx])
  pred_edge_sem[va_idx] = m_edge.predict([X_edge_sem[k] for k in va_idx])

print('=== ARESTAS COM pistas completas (como na predicao) ===')
print(classification_report(y_edge, pred_edge_com, zero_division=0))
print(
    '=== ARESTAS SEM pistas explicitas (generalizacao: PMI + TF-IDF contexto +'
    ' distancia + tipos) ==='
)
print(classification_report(y_edge, pred_edge_sem, zero_division=0))

pares de treino: 2039 | artigos: 40
=== ARESTAS COM pistas completas (como na predicao) ===
                  precision    recall  f1-score   support

  DIAGNOSED_WITH       0.96      0.96      0.96       210
FAMILIAR_HISTORY       0.33      0.17      0.22         6
         HISTORY       0.70      0.94      0.81        33
      LOCATED_IN       0.69      1.00      0.82        79
            NONE       1.00      0.93      0.96      1253
   PRESENTS_WITH       0.83      1.00      0.91       104
         REVEALS       0.64      1.00      0.78         7
           TAKES       0.96      0.99      0.97       139
  UNDERWENT_EXAM       0.92      1.00      0.96       208

        accuracy                           0.95      2039
       macro avg       0.78      0.89      0.82      2039
    weighted avg       0.95      0.95      0.95      2039

=== ARESTAS SEM pistas explicitas (generalizacao: PMI + TF-IDF contexto + distancia + tipos) ===
                  precision    recall  f1-score   supp

In [23]:
# Treina o modelo final de arestas com todo o conjunto de treino e salva em disco
edge_model = make_edge_model(seed=SEED).fit(X_edge_tr, y_edge)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(edge_model, MODEL_DIR / 'edge_model.joblib')
print('modelo de arestas salvo em', (MODEL_DIR / 'edge_model.joblib').relative_to(ROOT))
print()

# O que a Regressao Logistica Multinomial aprendeu: features de maior peso em cada classe de aresta
vec_e, clf_e = edge_model.steps[0][1], edge_model.steps[1][1]
nomes_e = vec_e.get_feature_names_out()
for k, classe in enumerate(clf_e.classes_):
    top = nomes_e[np.argsort(clf_e.coef_[k])[-8:][::-1]]
    print(f'{classe:18}', ', '.join(top))

modelo de arestas salvo em data/processed/model/edge_model.joblib

DIAGNOSED_WITH     pair_clause=Patient->Disease|same=True, pair=Patient->Disease, type_E2=Disease, pair_ppmi=Patient->Disease, pair_order=Patient->Disease|E1_before_E2, ctx_tfidf:the, pair_cue=Patient->Disease|DIAGNOSED, ctx_tfidf:the and
FAMILIAR_HISTORY   head_E2=epilepsy, pair_cue=Patient->Disease|FAMILY, cue=FAMILY, span_tfidf:family, head_E2=syndrome, span_tfidf:was negative, head_E2=conjunction, span_tfidf:negative
HISTORY            pair_cue=Patient->Disease|HISTORY, cue=HISTORY, head_E1=female, head_E2=insufficiency, span_tfidf:underlying, head_E2=fibrillation, head_E2=reflux, head_E2=ulceration
LOCATED_IN         pair_ppmi=Symptom->Anatomy, pair_ppmi=Disease->Anatomy, type_E2=Anatomy, cue=LOCATED, pair_dist=Symptom->Anatomy|close_1_3, pair_dist=Disease->Anatomy|med_4_8, pair_clause=Disease->Anatomy|same=True, pair=Disease->Anatomy
NONE               cue=NEGATION, span_tfidf:no, pair_cue=Patient->Disease|NEGATIO

In [24]:
from project02.models.classificar_arestas.edge_features import (
    predict_case_edges,
)

arestas_teste = []
todos_pares_teste = []

for row, ents in zip(test.to_dict('records'), test['entities']):
    cid = str(row['case_id'])
    aid = str(row[GROUP_COL])
    txt = str(row[TEXT_COL])

    # Todos os pares candidatos com distribuicao de probabilidades Softmax
    pares_all = predict_case_edges(edge_model, cid, aid, txt, ents, only_positive=False)
    todos_pares_teste.extend(pares_all)

    # Apenas as arestas positivas (pred_label != 'NONE') para construir o Grafo de Conhecimento
    arestas_pos = [
        {
            'e1_text': r['e1_text'],
            'type_E1': r['type_E1'],
            'edge_label': r['pred_label'],
            'e2_text': r['e2_text'],
            'type_E2': r['type_E2'],
            'span_between': r['span_between'],
            'token_dist': r['token_dist'],
            'pmi_head': r['pmi_head'],
            'confidence': r['pred_confidence'],
            'probabilities': r['probabilities'],
        }
        for r in pares_all if r['pred_label'] != 'NONE'
    ]
    arestas_teste.append(arestas_pos)

test['edges'] = arestas_teste
print('casos de teste processados:', len(test))
print('total de arestas extraidas no teste:', sum(len(x) for x in arestas_teste))
print('arestas predita por tipo:', pd.Series([
    e['edge_label'] for lista in arestas_teste for e in lista
]).value_counts().to_dict())

# Exibe a tabela de probabilidades Softmax para os pares do primeiro caso de teste (estilo Box 3)
df_box3 = pd.DataFrame([r for r in todos_pares_teste if r['case_id'] == str(test.iloc[0]['case_id'])])
if len(df_box3):
    cols_show = ['e1_text', 'type_E1', 'span_between', 'e2_text', 'type_E2', 'token_dist', 'pmi_head', 'pred_label', 'pred_confidence', 'probabilities']
    display(df_box3[cols_show].head(15))

casos de teste processados: 11
total de arestas extraidas no teste: 224
arestas predita por tipo: {'UNDERWENT_EXAM': 71, 'DIAGNOSED_WITH': 61, 'TAKES': 30, 'LOCATED_IN': 23, 'PRESENTS_WITH': 22, 'HISTORY': 10, 'REVEALS': 6, 'FAMILIAR_HISTORY': 1}


,e1_text,type_E1,span_between,e2_text,type_E2,token_dist,pmi_head,pred_label,pred_confidence,probabilities
0,woman,Patient,presented with a 3-day history of right flank ...,abdominal pain,Symptom,11,1.319,PRESENTS_WITH,0.9832,"{'DIAGNOSED_WITH': 0.0, 'FAMILIAR_HISTORY': 0...."
1,woman,Patient,presented with a 3-day history of right flank ...,nausea,Symptom,15,3.695,PRESENTS_WITH,0.9869,"{'DIAGNOSED_WITH': 0.0, 'FAMILIAR_HISTORY': 0...."
2,woman,Patient,presented with a 3-day history of right flank ...,constipation,Symptom,17,8.087,PRESENTS_WITH,0.9851,"{'DIAGNOSED_WITH': 0.0, 'FAMILIAR_HISTORY': 0...."
3,abdominal pain,Symptom,associated with,nausea,Symptom,2,3.514,NONE,0.9526,"{'DIAGNOSED_WITH': 0.0031, 'FAMILIAR_HISTORY':..."
4,abdominal pain,Symptom,associated with nausea and,constipation,Symptom,4,6.322,NONE,0.9999,"{'DIAGNOSED_WITH': 0.0, 'FAMILIAR_HISTORY': 0...."
5,nausea,Symptom,and,constipation,Symptom,1,7.746,NONE,0.9995,"{'DIAGNOSED_WITH': 0.0, 'FAMILIAR_HISTORY': 0...."
6,Patient,Patient,Her past medical family and medication history...,physical examination,Exam,12,1.299,UNDERWENT_EXAM,0.7049,"{'DIAGNOSED_WITH': 0.0, 'FAMILIAR_HISTORY': 0...."
7,Patient,Patient,She underwent contrast enhanced computed,tomography,Exam,5,1.089,UNDERWENT_EXAM,0.9841,"{'DIAGNOSED_WITH': 0.0005, 'FAMILIAR_HISTORY':..."
8,tomography,Exam,demonstrating a 6cm cystic lesion between the,stomach,Anatomy,7,4.097,NONE,0.9920,"{'DIAGNOSED_WITH': 0.0008, 'FAMILIAR_HISTORY':..."
9,tomography,Exam,demonstrating a 6cm cystic lesion between the ...,pancreas,Anatomy,13,5.075,NONE,0.9997,"{'DIAGNOSED_WITH': 0.0, 'FAMILIAR_HISTORY': 0...."


## 11. Extracao de Quantidades (Regex) + Associacao por Proximidade + Construcao do Grafo (`graph.py`)

Seguindo o diagrama de arquitetura do projeto:
1. **Identificador de Quantidades (`TOKEN_SEP_REGEX` + `QUANTITY_REGEX`):** Extrai valores numericos e unidades (`dimension`, `lenght`, `weight`, `volume`, `time`, `pressure`, `frequency`, `especial`) em cada sentenca.
2. **Associador de Quantidades (Algoritmo por Proximidade):** Vincula cada quantidade extraida a aresta positiva semanticamente compativel mais proxima na sentenca.
3. **Construtor do Grafo (`Graph`, `Node`, `Edge` de `graph.py`):** Instancia o grafo final de cada caso clinico e exporta os CSVs de nos e arestas.

In [32]:
from models.classificar_arestas.step3_attributes_and_graph import (
    associate_quantities_to_edges,
    build_case_graph,
    extract_sentence_quantities,
)

GRAPHS_DIR = PROJECT_DIR / 'data' / 'processed' / 'graphs'
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

grafos_teste = []
arestas_com_atributos_teste = []

for row, ents, pos_edges in zip(
    test.to_dict('records'), test['entities'], arestas_teste
):
  cid = str(row['case_id'])
  txt = str(row[TEXT_COL])

  # Constroi o objeto Graph (usando Node, Edge, Graph de graph.py)
  g = build_case_graph(
      case_row=row,
      raw_entities=ents,
      positive_edges=pos_edges,
      text_col=TEXT_COL,
  )
  grafos_teste.append(g)

  # Salva os CSVs de nos e arestas do caso usando o metodo save_to_csvs da classe Graph
  g.save_to_csvs(
      GRAPHS_DIR / f'{cid}_nodes.csv',
      GRAPHS_DIR / f'{cid}_edges.csv',
  )

  # Guarda resumo serializavel das arestas com atributos para o DataFrame de teste
  arestas_com_atributos_teste.append([
      {
          'from_node': e.from_node,
          'source_label': g.node(e.from_node).label,
          'relation': e.relation,
          'to_node': e.to_node,
          'target_label': g.node(e.to_node).label,
          'attributes': e.attributes,
      }
      for e in g.edges
  ])

test['graph_edges'] = arestas_com_atributos_teste

print(f'Grafos construidos: {len(grafos_teste)} (salvos em {GRAPHS_DIR.relative_to(ROOT)})')
print('Total de nos (soma dos casos de teste):', sum(g.len_nodes() for g in grafos_teste))
print('Total de arestas (soma dos casos de teste):', sum(g.len_edges() for g in grafos_teste))
print(
    'Arestas com atributos numericos/unidades:',
    sum(1 for g in grafos_teste for e in g.edges if 'value' in e.attributes),
)
print('\n================ EXEMPLO: GRAFO DO CASO 0 ================')
print(grafos_teste[0])

Grafos construidos: 11 (salvos em data/processed/graphs)
Total de nos (soma dos casos de teste): 268
Total de arestas (soma dos casos de teste): 189
Arestas com atributos numericos/unidades: 28

================ EXEMPLO: GRAFO DO CASO 0 ================
-------- Nodes --------
0: Node(type=Patient, label="Patient (PMC5137649_01)", attributes={'case_id': 'PMC5137649_01', 'age': '44.0', 'gender': 'Female'}, pos_in_case=0)
1: Node(type=Symptom, label="abdominal pain", attributes={}, pos_in_case=14)
2: Node(type=Symptom, label="nausea", attributes={}, pos_in_case=18)
3: Node(type=Symptom, label="constipation", attributes={}, pos_in_case=20)
4: Node(type=Exam, label="physical examination", attributes={}, pos_in_case=33)
5: Node(type=Exam, label="tomography", attributes={}, pos_in_case=42)
6: Node(type=Anatomy, label="stomach", attributes={}, pos_in_case=50)
7: Node(type=Anatomy, label="pancreas", attributes={}, pos_in_case=56)
8: Node(type=Disease, label="pancreatic", attributes={}, pos_in_

In [33]:
# Tabela consolidada das arestas que receberam quantidades (valor + unidade) no conjunto de teste
linhas_attr = []
for row, g in zip(test.to_dict('records'), grafos_teste):
    cid = row['case_id']
    for idx, e in enumerate(g.edges):
        if 'value' in e.attributes:
            linhas_attr.append({
                'case_id': cid,
                'edge_idx': idx,
                'source': f"{g.node(e.from_node).label} ({g.node(e.from_node).node_type})",
                'relation': e.relation,
                'target': f"{g.node(e.to_node).label} ({g.node(e.to_node).node_type})",
                'value': e.attributes.get('value', ''),
                'unit': e.attributes.get('unit', ''),
                'rule_type': e.attributes.get('type', ''),
                'confidence': e.attributes.get('confidence', ''),
            })

df_attr = pd.DataFrame(linhas_attr)
display(df_attr.head(25) if len(df_attr) else 'Nenhuma quantidade associada.')

,case_id,edge_idx,source,relation,target,value,unit,rule_type,confidence
0,PMC5137649_01,0,Patient (PMC5137649_01) (Patient),PRESENTS_WITH,abdominal pain (Symptom),44,year,time,0.9832
1,PMC5137649_01,5,Patient (PMC5137649_01) (Patient),DIAGNOSED_WITH,pancreatic (Disease),6cm x 9cm,cm,dimension,0.8895
2,PMC5137649_01,6,Patient (PMC5137649_01) (Patient),DIAGNOSED_WITH,cyst (Disease),6,cm,lenght,0.716
3,PMC5833205_01,0,Patient (PMC5833205_01) (Patient),PRESENTS_WITH,edema (Symptom),2,weeks,time,0.9745
4,PMC5833205_01,4,Patient (PMC5833205_01) (Patient),UNDERWENT_EXAM,urinalysis (Exam),1.8,g/dL,weight,0.9961
5,PMC5833205_01,6,Patient (PMC5833205_01) (Patient),DIAGNOSED_WITH,anemia (Disease),6.4,mg/dL,weight,0.9373
6,PMC5833205_01,8,Patient (PMC5833205_01) (Patient),DIAGNOSED_WITH,hypoproteinemia (Disease),4.9,ng/L,weight,0.9769
7,PMC5833205_01,9,Patient (PMC5833205_01) (Patient),UNDERWENT_EXAM,occult blood (Exam),1.13,mg/g,weight,0.9956
8,PMC5833205_01,12,Patient (PMC5833205_01) (Patient),PRESENTS_WITH,edema (Symptom),1,week,time,0.9896
9,PMC9529523_01,0,Patient (PMC9529523_01) (Patient),HISTORY,cardiomyopathy (Disease),41,year,time,0.9956
